In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import sklearn
import scipy
from pathlib import Path
import json
from torch.utils.data import Dataset, DataLoader

p  = Path('data')
arr = []
for file in sorted(p.iterdir()):
    df = pd.read_csv(file)
    arr.append(df)

train_data = pd.concat(arr[1:], ignore_index= True)
validate_data = arr[0][:arr[0].shape[0]//2]
test_data = arr[0][arr[0].shape[0]//2:].reset_index(drop=True)

features_train_data = train_data.drop(columns = ['eye_state'])
labels_train_data = train_data['eye_state']

features_validate_data = validate_data.drop(columns = ['eye_state'])
labels_validate_data = validate_data['eye_state']

features_test_data = test_data.drop(columns = ['eye_state'])
labels_test_data = test_data['eye_state']


def clean(s, threshold):
    mean = s.mean()
    s = s.copy()
    s[np.abs(s - mean) > threshold] = mean
    return s

def normalize(df):
    return (df - df.mean()) / df.std()

def preprocess(df, threshold):
    df = df.copy()
    df = df.apply(clean, threshold=threshold)
    df = normalize(df)
    return df
    
features_train_data = preprocess(features_train_data, threshold=250)
features_validate_data = preprocess(features_validate_data, threshold=250)
features_test_data = preprocess(features_test_data, threshold=250)

In [5]:
class customDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features.iloc[idx].values, self.labels.iloc[idx]
train_dataset = customDataset(features_train_data, labels_train_data)
valid_dataset = customDataset(features_validate_data, labels_validate_data)
test_dataset = customDataset(features_test_data, labels_test_data)

In [7]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [9]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [ ]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = torch.nn.Flatten()
        self.layers = torch.nn.Sequential()
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

_IncompleteInputError: incomplete input (1209640577.py, line 7)